# 🎓 Personalized Learning Path Recommendation System Using AI
### Project BT3116 | IILM University, Greater Noida | Session 2025-26
**Team:** Swastik Srivastava · Stuty Singh · Suryansh Singh · Suprit Dubey · Tanisha Priya  
**Guide:** Dr. Jaswinder Singh Walia

---
> This notebook implements a **Content-Based Filtering** recommender system using **TF-IDF Vectorization** and **Cosine Similarity**, with a beautiful **Gradio UI**, **DAG path visualization**, and evaluation via **NDCG@K, MAP@K, Precision@K**.

## ⚙️ Step 0 — Install Dependencies

In [ ]:
!pip install -q gradio scikit-learn pandas numpy networkx matplotlib plotly

## 📦 Step 1 — Imports & Dataset

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import warnings
import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
print("✅ All libraries loaded successfully!")

## 🗃️ Step 2 — Course Dataset (20 Courses × 7 Attributes)

In [ ]:
courses_data = [
    {
        "name": "Python Programming Fundamentals",
        "category": "Programming",
        "level": "Beginner",
        "bloom_level": 1,
        "skills": "python variables loops functions data types debugging",
        "duration": 20,
        "prerequisites": ""
    },
    {
        "name": "Data Structures and Algorithms",
        "category": "Programming",
        "level": "Intermediate",
        "bloom_level": 3,
        "skills": "python arrays linked lists trees graphs sorting searching recursion complexity",
        "duration": 35,
        "prerequisites": "Python Programming Fundamentals"
    },
    {
        "name": "Statistics for Data Science",
        "category": "Data Science",
        "level": "Beginner",
        "bloom_level": 2,
        "skills": "statistics probability distributions hypothesis testing mean variance regression",
        "duration": 25,
        "prerequisites": ""
    },
    {
        "name": "Machine Learning Fundamentals",
        "category": "AI/ML",
        "level": "Intermediate",
        "bloom_level": 3,
        "skills": "machine learning supervised unsupervised classification regression clustering scikit-learn",
        "duration": 40,
        "prerequisites": "Python Programming Fundamentals Statistics for Data Science"
    },
    {
        "name": "Deep Learning with TensorFlow",
        "category": "AI/ML",
        "level": "Advanced",
        "bloom_level": 5,
        "skills": "deep learning neural networks tensorflow keras CNNs RNNs backpropagation GPU",
        "duration": 50,
        "prerequisites": "Machine Learning Fundamentals"
    },
    {
        "name": "Natural Language Processing",
        "category": "AI/ML",
        "level": "Advanced",
        "bloom_level": 5,
        "skills": "NLP text processing tokenization transformers BERT sentiment analysis text classification",
        "duration": 45,
        "prerequisites": "Deep Learning with TensorFlow"
    },
    {
        "name": "HTML CSS Web Basics",
        "category": "Web Development",
        "level": "Beginner",
        "bloom_level": 1,
        "skills": "html css web design layouts flexbox grid responsive design",
        "duration": 18,
        "prerequisites": ""
    },
    {
        "name": "JavaScript Essentials",
        "category": "Web Development",
        "level": "Beginner",
        "bloom_level": 2,
        "skills": "javascript DOM events async promises fetch API ES6 web programming",
        "duration": 22,
        "prerequisites": "HTML CSS Web Basics"
    },
    {
        "name": "React.js Frontend Development",
        "category": "Web Development",
        "level": "Intermediate",
        "bloom_level": 3,
        "skills": "react components hooks state props JSX frontend SPA routing redux",
        "duration": 30,
        "prerequisites": "JavaScript Essentials"
    },
    {
        "name": "Node.js Backend Development",
        "category": "Web Development",
        "level": "Intermediate",
        "bloom_level": 3,
        "skills": "node.js express REST API backend server authentication JWT database MongoDB",
        "duration": 28,
        "prerequisites": "JavaScript Essentials"
    },
    {
        "name": "SQL and Database Design",
        "category": "Data Science",
        "level": "Beginner",
        "bloom_level": 2,
        "skills": "SQL databases tables queries joins normalization relational CRUD postgres mysql",
        "duration": 20,
        "prerequisites": ""
    },
    {
        "name": "Data Analysis with Pandas",
        "category": "Data Science",
        "level": "Intermediate",
        "bloom_level": 3,
        "skills": "pandas numpy data wrangling EDA analysis visualization matplotlib seaborn data cleaning",
        "duration": 25,
        "prerequisites": "Python Programming Fundamentals Statistics for Data Science"
    },
    {
        "name": "Computer Vision with OpenCV",
        "category": "AI/ML",
        "level": "Advanced",
        "bloom_level": 5,
        "skills": "computer vision OpenCV image processing detection classification object recognition CNN",
        "duration": 42,
        "prerequisites": "Deep Learning with TensorFlow"
    },
    {
        "name": "Cloud Computing with AWS",
        "category": "Cloud Computing",
        "level": "Intermediate",
        "bloom_level": 3,
        "skills": "AWS cloud EC2 S3 Lambda deployment DevOps microservices containerization",
        "duration": 32,
        "prerequisites": "Python Programming Fundamentals"
    },
    {
        "name": "Docker and Kubernetes",
        "category": "Cloud Computing",
        "level": "Advanced",
        "bloom_level": 4,
        "skills": "docker kubernetes containers orchestration CI/CD deployment scaling DevOps microservices",
        "duration": 35,
        "prerequisites": "Cloud Computing with AWS"
    },
    {
        "name": "MLOps and Model Deployment",
        "category": "AI/ML",
        "level": "Advanced",
        "bloom_level": 5,
        "skills": "MLOps model deployment pipeline CI/CD monitoring drift docker kubernetes FastAPI",
        "duration": 38,
        "prerequisites": "Machine Learning Fundamentals Docker and Kubernetes"
    },
    {
        "name": "Cybersecurity Essentials",
        "category": "Security",
        "level": "Beginner",
        "bloom_level": 2,
        "skills": "cybersecurity networking firewalls encryption authentication vulnerabilities OWASP",
        "duration": 22,
        "prerequisites": ""
    },
    {
        "name": "Ethical Hacking and Penetration Testing",
        "category": "Security",
        "level": "Advanced",
        "bloom_level": 5,
        "skills": "ethical hacking penetration testing Kali Linux CTF vulnerability exploitation security audit",
        "duration": 45,
        "prerequisites": "Cybersecurity Essentials"
    },
    {
        "name": "Reinforcement Learning",
        "category": "AI/ML",
        "level": "Advanced",
        "bloom_level": 6,
        "skills": "reinforcement learning Q-learning policy gradient reward agent environment OpenAI Gym",
        "duration": 48,
        "prerequisites": "Deep Learning with TensorFlow"
    },
    {
        "name": "Full Stack Web Development",
        "category": "Web Development",
        "level": "Advanced",
        "bloom_level": 5,
        "skills": "fullstack MERN react node mongodb REST API deployment authentication testing",
        "duration": 55,
        "prerequisites": "React.js Frontend Development Node.js Backend Development SQL and Database Design"
    }
]

df = pd.DataFrame(courses_data)
print(f"✅ Dataset loaded: {len(df)} courses × {len(df.columns)} attributes")
df.head()

## 🧠 Step 3 — Feature Engineering & TF-IDF Vectorization

In [ ]:
LEVEL_WEIGHT = {"Beginner": "beginner beginner", "Intermediate": "intermediate intermediate intermediate", "Advanced": "advanced advanced advanced advanced"}
BLOOM_LABELS = {1: "remember", 2: "understand", 3: "apply", 4: "analyze", 5: "evaluate", 6: "create"}

def build_feature_string(row):
    """Weighted multi-attribute feature string."""
    category  = (row["category"] + " ") * 3
    level_str = LEVEL_WEIGHT.get(row["level"], row["level"])
    bloom_str = (BLOOM_LABELS.get(row["bloom_level"], "") + " ") * 2
    skills    = row["skills"] * 2           # skills weighted high
    prereqs   = row["prerequisites"].replace(",", " ")
    duration  = "short" if row["duration"] < 25 else ("medium" if row["duration"] < 40 else "long")
    return f"{category} {level_str} {bloom_str} {skills} {prereqs} {duration}"

df["feature_string"] = df.apply(build_feature_string, axis=1)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=500, stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["feature_string"])

# Cosine Similarity Matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

course_index = pd.Series(df.index, index=df["name"]).drop_duplicates()

print(f"✅ TF-IDF Matrix: {tfidf_matrix.shape}")
print(f"✅ Cosine Similarity Matrix: {cosine_sim.shape}")

## 🔍 Step 4 — Recommendation Engine with Explainability

In [ ]:
LEVEL_ORDER = {"Beginner": 1, "Intermediate": 2, "Advanced": 3}

def get_recommendations(course_name, top_k=5, filter_category=None, filter_level=None):
    """
    Returns top-K course recommendations with similarity scores and natural language explanations.
    """
    if course_name not in course_index:
        return None, f"❌ Course '{course_name}' not found in dataset."

    idx = course_index[course_name]
    current = df.iloc[idx]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [(i, s) for i, s in sim_scores if i != idx]  # exclude self

    results = []
    for i, score in sim_scores:
        candidate = df.iloc[i]

        # Optional filters
        if filter_category and filter_category != "All" and candidate["category"] != filter_category:
            continue
        if filter_level and filter_level != "All" and candidate["level"] != filter_level:
            continue

        # Build natural language explanation
        reasons = []
        if candidate["category"] == current["category"]:
            reasons.append(f"same domain ({candidate['category']})")
        current_skills = set(current["skills"].split())
        cand_skills    = set(candidate["skills"].split())
        shared = current_skills & cand_skills
        if shared:
            reasons.append(f"shared skills: {', '.join(list(shared)[:3])}")
        level_diff = LEVEL_ORDER.get(candidate["level"], 2) - LEVEL_ORDER.get(current["level"], 2)
        if level_diff == 1:
            reasons.append("natural difficulty progression (+1 level)")
        elif level_diff == 0:
            reasons.append("same difficulty level")
        if current["name"] in candidate["prerequisites"]:
            reasons.append("🔗 direct prerequisite satisfied")

        explanation = "Recommended because: " + "; ".join(reasons) if reasons else "Content similarity match"

        results.append({
            "rank": len(results) + 1,
            "name": candidate["name"],
            "category": candidate["category"],
            "level": candidate["level"],
            "duration": f"{candidate['duration']}h",
            "similarity": round(score * 100, 1),
            "explanation": explanation,
            "prerequisites": candidate["prerequisites"] if candidate["prerequisites"] else "None"
        })

        if len(results) >= top_k:
            break

    return results, None

# Quick test
recs, err = get_recommendations("Machine Learning Fundamentals", top_k=5)
if recs:
    print("\n🎯 Sample Recommendations for: Machine Learning Fundamentals")
    for r in recs:
        print(f"  #{r['rank']} {r['name']} [{r['level']}] — {r['similarity']}% match")
        print(f"      💡 {r['explanation']}")

## 🕸️ Step 5 — DAG Learning Path Generator

In [ ]:
def build_dag(start_course, max_depth=4):
    """
    Builds a Directed Acyclic Graph (DAG) showing the learning path from a starting course.
    Returns a Plotly figure.
    """
    G = nx.DiGraph()

    def add_path(course_name, depth):
        if depth > max_depth or course_name not in course_index:
            return
        idx = course_index[course_name]
        G.add_node(course_name,
                   level=df.iloc[idx]["level"],
                   category=df.iloc[idx]["category"])
        recs, _ = get_recommendations(course_name, top_k=3)
        if recs:
            for r in recs[:2]:  # add top-2 edges
                next_c = r["name"]
                if not G.has_edge(course_name, next_c):
                    G.add_edge(course_name, next_c, weight=r["similarity"])
                    add_path(next_c, depth + 1)

    add_path(start_course, 0)

    if len(G.nodes) == 0:
        return None

    # Layout
    try:
        pos = nx.spring_layout(G, k=2.5, seed=42)
    except:
        pos = nx.shell_layout(G)

    LEVEL_COLOR = {
        "Beginner":     "#22c55e",
        "Intermediate": "#f59e0b",
        "Advanced":     "#ef4444"
    }

    edge_x, edge_y = [], []
    for u, v in G.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

    edge_trace = go.Scatter(x=edge_x, y=edge_y, mode="lines",
                            line=dict(width=1.5, color="#94a3b8"),
                            hoverinfo="none")

    node_x = [pos[n][0] for n in G.nodes()]
    node_y = [pos[n][1] for n in G.nodes()]
    node_colors = [LEVEL_COLOR.get(G.nodes[n].get("level", "Beginner"), "#6366f1") for n in G.nodes()]
    node_text  = [f"<b>{n}</b><br>{G.nodes[n].get('category','')}<br>{G.nodes[n].get('level','')}" for n in G.nodes()]
    node_labels = [n if n == start_course else (n[:22]+".." if len(n)>24 else n) for n in G.nodes()]

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode="markers+text",
        text=node_labels,
        textposition="top center",
        textfont=dict(size=10, color="white"),
        hovertext=node_text,
        hoverinfo="text",
        marker=dict(size=22, color=node_colors,
                    line=dict(width=2, color="white"))
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            title=dict(text=f"🗺️ Learning Path DAG — Starting: {start_course}",
                       font=dict(size=14, color="#e2e8f0")),
            paper_bgcolor="#0f172a",
            plot_bgcolor="#0f172a",
            font=dict(color="#e2e8f0"),
            showlegend=False,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            margin=dict(l=20, r=20, t=60, b=20),
            height=480
        )
    )

    # Legend
    for level, color in LEVEL_COLOR.items():
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=12, color=color),
            name=level, showlegend=True
        ))

    return fig

# Quick test
fig = build_dag("Python Programming Fundamentals")
if fig:
    fig.show()
    print("✅ DAG generated successfully!")

## 📊 Step 6 — Evaluation Metrics (NDCG@K, MAP@K, Precision@K)

In [ ]:
def build_ground_truth():
    """Ground truth: same category OR prerequisite chains = relevant."""
    gt = {}
    for _, row in df.iterrows():
        relevant = []
        for _, other in df.iterrows():
            if other["name"] == row["name"]: continue
            if other["category"] == row["category"] or row["name"] in other["prerequisites"]:
                relevant.append(other["name"])
        gt[row["name"]] = relevant
    return gt

ground_truth = build_ground_truth()

def precision_at_k(recommended, relevant, k):
    r = recommended[:k]
    return len([x for x in r if x in relevant]) / k if k > 0 else 0.0

def average_precision_at_k(recommended, relevant, k):
    hits, score = 0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant), k) if relevant else 0.0

def ndcg_at_k(recommended, relevant, k):
    dcg, idcg = 0.0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)
    for i in range(min(len(relevant), k)):
        idcg += 1 / np.log2(i + 2)
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_system(k_values=[1, 3, 5, 7, 10]):
    results = {k: {"precision": [], "map": [], "ndcg": []} for k in k_values}
    for course in df["name"]:
        recs, err = get_recommendations(course, top_k=max(k_values))
        if not recs: continue
        rec_names = [r["name"] for r in recs]
        relevant  = ground_truth.get(course, [])
        for k in k_values:
            results[k]["precision"].append(precision_at_k(rec_names, relevant, k))
            results[k]["map"].append(average_precision_at_k(rec_names, relevant, k))
            results[k]["ndcg"].append(ndcg_at_k(rec_names, relevant, k))

    print("\n" + "═"*70)
    print(f"{'K':>4} │ {'Precision@K':>12} │ {'MAP@K':>10} │ {'NDCG@K':>10}")
    print("═"*70)
    for k in k_values:
        p  = np.mean(results[k]["precision"])
        m  = np.mean(results[k]["map"])
        n  = np.mean(results[k]["ndcg"])
        print(f"{k:>4} │ {p:>12.4f} │ {m:>10.4f} │ {n:>10.4f}")
    print("═"*70)
    return results

eval_results = evaluate_system()
print("\n✅ Evaluation complete!")

## 📈 Step 7 — Evaluation Visualization

In [ ]:
k_values = [1, 3, 5, 7, 10]
prec  = [np.mean(eval_results[k]["precision"]) for k in k_values]
maps  = [np.mean(eval_results[k]["map"])       for k in k_values]
ndcgs = [np.mean(eval_results[k]["ndcg"])      for k in k_values]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor("#0f172a")

metrics = [(prec, "Precision@K", "#22d3ee"), (maps, "MAP@K", "#a78bfa"), (ndcgs, "NDCG@K", "#34d399")]
for ax, (vals, title, color) in zip(axes, metrics):
    ax.set_facecolor("#1e293b")
    ax.plot(k_values, vals, color=color, linewidth=2.5, marker="o", markersize=8)
    ax.fill_between(k_values, vals, alpha=0.2, color=color)
    ax.set_title(title, color="white", fontsize=13, pad=12)
    ax.set_xlabel("K", color="#94a3b8"); ax.set_ylabel("Score", color="#94a3b8")
    ax.tick_params(colors="#94a3b8")
    for spine in ax.spines.values(): spine.set_edgecolor("#334155")
    ax.grid(True, color="#334155", linestyle="--", alpha=0.5)
    ax.set_ylim(0, 1.05)

plt.suptitle("📊 Recommender System Evaluation Metrics", color="white", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()
print("✅ Evaluation charts rendered!")

## 🌟 Step 8 — Gradio Interface (Beautiful Dark Theme)

In [ ]:
# ─── Custom CSS ───────────────────────────────────────────────────────────────
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600;700&display=swap');

:root {
  --bg-primary:   #0a0f1e;
  --bg-card:      #0f1a2e;
  --bg-input:     #111827;
  --accent-blue:  #38bdf8;
  --accent-violet:#a78bfa;
  --accent-green: #34d399;
  --accent-amber: #fbbf24;
  --text-primary: #f1f5f9;
  --text-muted:   #94a3b8;
  --border:       #1e3a5f;
  --glow:         0 0 20px rgba(56, 189, 248, 0.3);
}

body, .gradio-container {
  background: var(--bg-primary) !important;
  font-family: 'DM Sans', sans-serif !important;
  color: var(--text-primary) !important;
}

/* Header */
.app-header {
  background: linear-gradient(135deg, #0a1628 0%, #0f1f3d 50%, #0a1628 100%);
  border: 1px solid var(--border);
  border-radius: 16px;
  padding: 32px 40px;
  margin-bottom: 24px;
  position: relative;
  overflow: hidden;
}
.app-header::before {
  content: '';
  position: absolute;
  top: -50%; left: -50%;
  width: 200%; height: 200%;
  background: radial-gradient(ellipse at 60% 40%, rgba(56,189,248,0.08) 0%, transparent 60%),
              radial-gradient(ellipse at 30% 70%, rgba(167,139,250,0.07) 0%, transparent 50%);
  pointer-events: none;
}
.app-title {
  font-family: 'Space Mono', monospace !important;
  font-size: 26px !important;
  font-weight: 700 !important;
  background: linear-gradient(90deg, #38bdf8, #a78bfa, #34d399);
  -webkit-background-clip: text;
  -webkit-text-fill-color: transparent;
  background-clip: text;
  margin: 0 !important;
  letter-spacing: -0.5px;
}
.app-subtitle {
  color: var(--text-muted) !important;
  font-size: 13px !important;
  margin-top: 6px !important;
  letter-spacing: 0.5px;
}

/* Panels */
.gradio-group, .gr-group {
  background: var(--bg-card) !important;
  border: 1px solid var(--border) !important;
  border-radius: 12px !important;
}

/* Inputs */
select, .gr-dropdown select, textarea, input[type=range] {
  background: var(--bg-input) !important;
  border: 1px solid var(--border) !important;
  color: var(--text-primary) !important;
  border-radius: 8px !important;
  font-family: 'DM Sans', sans-serif !important;
}

/* Labels */
label, .gr-label, .gr-form label {
  color: var(--accent-blue) !important;
  font-size: 12px !important;
  font-weight: 600 !important;
  letter-spacing: 0.8px !important;
  text-transform: uppercase !important;
}

/* Buttons */
button.primary {
  background: linear-gradient(135deg, #1d4ed8, #7c3aed) !important;
  border: none !important;
  border-radius: 10px !important;
  font-family: 'Space Mono', monospace !important;
  font-size: 13px !important;
  font-weight: 700 !important;
  letter-spacing: 1px !important;
  padding: 12px 28px !important;
  transition: all 0.25s ease !important;
  box-shadow: 0 4px 20px rgba(124, 58, 237, 0.4) !important;
}
button.primary:hover {
  transform: translateY(-2px) !important;
  box-shadow: 0 8px 30px rgba(56, 189, 248, 0.5) !important;
}
button.secondary {
  background: transparent !important;
  border: 1px solid var(--border) !important;
  color: var(--text-muted) !important;
  border-radius: 10px !important;
  font-size: 12px !important;
}

/* Output HTML card */
.rec-card {
  background: linear-gradient(135deg, #0f1a2e 0%, #111827 100%);
  border: 1px solid #1e3a5f;
  border-radius: 12px;
  padding: 18px 22px;
  margin-bottom: 14px;
  position: relative;
  overflow: hidden;
  transition: border-color 0.2s;
}
.rec-card::before {
  content: '';
  position: absolute;
  left: 0; top: 0; bottom: 0;
  width: 4px;
  background: linear-gradient(180deg, #38bdf8, #a78bfa);
  border-radius: 4px 0 0 4px;
}
.rec-rank   { font-family: 'Space Mono', monospace; font-size: 28px; font-weight: 700; color: #1e3a5f; }
.rec-title  { font-weight: 700; font-size: 16px; color: #f1f5f9; }
.rec-meta   { font-size: 12px; color: #64748b; margin: 6px 0; }
.rec-badge  { display: inline-block; padding: 3px 10px; border-radius: 20px; font-size: 11px; font-weight: 600; margin-right: 6px; }
.badge-beg  { background: rgba(34,197,94,0.15);  color: #22c55e;  border: 1px solid #22c55e40; }
.badge-int  { background: rgba(245,158,11,0.15); color: #f59e0b;  border: 1px solid #f59e0b40; }
.badge-adv  { background: rgba(239,68,68,0.15);  color: #ef4444;  border: 1px solid #ef444440; }
.badge-cat  { background: rgba(56,189,248,0.10); color: #38bdf8;  border: 1px solid #38bdf840; }
.sim-bar-wrap { background: #1e293b; border-radius: 20px; height: 8px; margin: 10px 0; overflow: hidden; }
.sim-bar-fill { height: 100%; border-radius: 20px; background: linear-gradient(90deg, #38bdf8, #a78bfa); transition: width 0.6s ease; }
.rec-explain { font-size: 12px; color: #7c9bc2; margin-top: 8px; }
.rec-prereq  { font-size: 11px; color: #475569; margin-top: 5px; }

.stats-grid {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 12px;
  margin-bottom: 20px;
}
.stat-card {
  background: linear-gradient(135deg, #0f1a2e, #111827);
  border: 1px solid #1e3a5f;
  border-radius: 10px;
  padding: 16px;
  text-align: center;
}
.stat-val  { font-family: 'Space Mono', monospace; font-size: 24px; font-weight: 700; }
.stat-lbl  { font-size: 11px; color: #64748b; text-transform: uppercase; letter-spacing: 0.5px; margin-top: 4px; }

.section-header {
  font-family: 'Space Mono', monospace;
  font-size: 11px;
  color: #38bdf8;
  letter-spacing: 2px;
  text-transform: uppercase;
  margin-bottom: 14px;
  padding-bottom: 8px;
  border-bottom: 1px solid #1e3a5f;
}
"""

# ─── Helper: build HTML output ─────────────────────────────────────────────
LEVEL_BADGE = {"Beginner": "badge-beg", "Intermediate": "badge-int", "Advanced": "badge-adv"}

def build_results_html(course_name, recs):
    if not recs:
        return "<p style='color:#ef4444;'>No recommendations found.</p>"

    avg_sim = np.mean([r["similarity"] for r in recs])
    categories = set(r["category"] for r in recs)

    html = f"""
    <div style='padding:4px'>
      <div class='section-header'>📊 Result Summary — {course_name}</div>
      <div class='stats-grid'>
        <div class='stat-card'>
          <div class='stat-val' style='color:#38bdf8'>{len(recs)}</div>
          <div class='stat-lbl'>Recommendations</div>
        </div>
        <div class='stat-card'>
          <div class='stat-val' style='color:#a78bfa'>{avg_sim:.1f}%</div>
          <div class='stat-lbl'>Avg. Similarity</div>
        </div>
        <div class='stat-card'>
          <div class='stat-val' style='color:#34d399'>{len(categories)}</div>
          <div class='stat-lbl'>Domains Covered</div>
        </div>
      </div>
      <div class='section-header'>🎯 Ranked Recommendations</div>
    """

    for r in recs:
        badge_cls = LEVEL_BADGE.get(r["level"], "badge-int")
        bar_width = max(5, int(r["similarity"]))
        html += f"""
        <div class='rec-card'>
          <div style='display:flex; align-items:center; gap:14px;'>
            <div class='rec-rank'>#{r['rank']:02d}</div>
            <div style='flex:1'>
              <div class='rec-title'>{r['name']}</div>
              <div class='rec-meta'>
                <span class='rec-badge {badge_cls}'>{r['level']}</span>
                <span class='rec-badge badge-cat'>{r['category']}</span>
                <span style='color:#475569'>⏱ {r['duration']}</span>
              </div>
              <div class='sim-bar-wrap'>
                <div class='sim-bar-fill' style='width:{bar_width}%'></div>
              </div>
              <div style='display:flex; justify-content:space-between; font-size:11px; color:#475569;'>
                <span>Similarity Match</span>
                <span style='color:#38bdf8; font-family:Space Mono,monospace; font-weight:700'>{r['similarity']}%</span>
              </div>
              <div class='rec-explain'>💡 {r['explanation']}</div>
              <div class='rec-prereq'>🔗 Prerequisites: {r['prerequisites']}</div>
            </div>
          </div>
        </div>
        """

    html += "</div>"
    return html

print("✅ CSS and HTML builders ready!")

In [ ]:
# ─── Gradio App ──────────────────────────────────────────────────────────────
ALL_COURSES   = sorted(df["name"].tolist())
ALL_CATS      = ["All"] + sorted(df["category"].unique().tolist())
ALL_LEVELS    = ["All", "Beginner", "Intermediate", "Advanced"]

HEADER_HTML = """
<div class='app-header'>
  <div class='app-title'>⚡ AI Learning Path Recommender</div>
  <div class='app-subtitle'>
    PROJECT BT3116 &nbsp;·&nbsp; IILM UNIVERSITY, GREATER NOIDA &nbsp;·&nbsp;
    TF-IDF + COSINE SIMILARITY ENGINE
  </div>
</div>
"""

def recommend_and_visualize(course_name, top_k, filter_cat, filter_level):
    recs, err = get_recommendations(course_name, int(top_k), filter_cat, filter_level)
    if err:
        return f"<p style='color:#ef4444;'>{err}</p>", None
    html = build_results_html(course_name, recs)
    dag  = build_dag(course_name, max_depth=3)
    return html, dag

with gr.Blocks(css=CUSTOM_CSS, title="AI Learning Path Recommender") as demo:

    gr.HTML(HEADER_HTML)

    with gr.Row():
        # ── Left panel — controls
        with gr.Column(scale=1):
            with gr.Group():
                gr.HTML("<div style='padding:16px 18px 4px; font-family:Space Mono,monospace; font-size:11px; color:#38bdf8; letter-spacing:2px; text-transform:uppercase;'>🎛  Configuration</div>")

                course_dd = gr.Dropdown(
                    choices=ALL_COURSES,
                    value=ALL_COURSES[0],
                    label="Select Your Current Course",
                    info="The course you have completed or are currently taking"
                )
                k_slider = gr.Slider(
                    minimum=1, maximum=10, step=1, value=5,
                    label="Number of Recommendations (Top-K)"
                )
                cat_dd = gr.Dropdown(
                    choices=ALL_CATS, value="All",
                    label="Filter by Domain Category"
                )
                level_dd = gr.Dropdown(
                    choices=ALL_LEVELS, value="All",
                    label="Filter by Difficulty Level"
                )
                submit_btn = gr.Button("🚀  Generate My Learning Path", variant="primary")
                clear_btn  = gr.Button("↺  Reset", variant="secondary")

            with gr.Group():
                gr.HTML("""
                <div style='padding:14px 18px;'>
                  <div style='font-family:Space Mono,monospace; font-size:10px; color:#38bdf8; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;'>ℹ️  How It Works</div>
                  <div style='font-size:12px; color:#64748b; line-height:1.8;'>
                    <b style='color:#94a3b8;'>1. Feature Engineering</b><br>Multi-attribute weighted feature strings<br><br>
                    <b style='color:#94a3b8;'>2. TF-IDF Vectorization</b><br>Bigram TF-IDF with sublinear scaling<br><br>
                    <b style='color:#94a3b8;'>3. Cosine Similarity</b><br>20×20 similarity matrix<br><br>
                    <b style='color:#94a3b8;'>4. DAG Path Generator</b><br>NetworkX sequential learning graph<br><br>
                    <b style='color:#94a3b8;'>5. Evaluation</b><br>NDCG@K · MAP@K · Precision@K
                  </div>
                </div>
                """)

        # ── Right panel — outputs
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("📋 Recommendations"):
                    rec_output = gr.HTML(label="Recommendations")
                with gr.TabItem("🗺️ Learning Path DAG"):
                    dag_output = gr.Plot(label="Learning Path Graph")

    submit_btn.click(
        fn=recommend_and_visualize,
        inputs=[course_dd, k_slider, cat_dd, level_dd],
        outputs=[rec_output, dag_output]
    )

    def reset():
        return ALL_COURSES[0], 5, "All", "All", "", None

    clear_btn.click(
        fn=reset,
        outputs=[course_dd, k_slider, cat_dd, level_dd, rec_output, dag_output]
    )

    gr.HTML("""
    <div style='text-align:center; padding:20px 0 8px; color:#1e3a5f; font-size:11px; font-family:Space Mono,monospace; letter-spacing:1px;'>
      IILM UNIVERSITY · SCHOOL OF COMPUTER SCIENCE & ENGINEERING ·
      SWASTIK · STUTY · SURYANSH · SUPRIT · TANISHA · GUIDE: DR. J.S. WALIA
    </div>
    """)

print("✅ Gradio app configured. Launching now...")
demo.launch(share=True, debug=False)

---
## 🏁 Summary

| Component | Details |
|---|---|
| Dataset | 20 courses × 7 attributes |
| Algorithm | Content-Based Filtering |
| Vectorization | TF-IDF (bigrams, sublinear TF, 500 features) |
| Similarity | Cosine Similarity (20×20 matrix) |
| UI | Gradio (dark theme, tabs, DAG) |
| Evaluation | NDCG@K · MAP@K · Precision@K (K=1,3,5,7,10) |
| Visualization | Plotly DAG, Matplotlib metrics |

> **Project BT3116 · IILM University, Greater Noida · Session 2025-26**